huggingface-colab-exploration

In [3]:
!pip install -q huggingface_hub

from huggingface_hub import login
login()

In [4]:
!nvidia-smi

Wed Sep 23 03:17:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Qunatization

In [3]:
prompt = "The future of AI engineering is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = quantized_model.generate(
        **inputs,
        max_new_tokens=15,
        output_scores=True,
        return_dict_in_generate=True,
    )

generated_ids = output.sequences[0][inputs["input_ids"].shape[1]:]

for i, token_id in enumerate(generated_ids):
    token_str = tokenizer.decode(token_id)
    print(f"Step {i+1}: generated '{token_str}'")

Step 1: generated ' bright'
Step 2: generated ','
Step 3: generated ' but'
Step 4: generated ' there'
Step 5: generated ' are'
Step 6: generated ' still'
Step 7: generated ' a'
Step 8: generated ' lot'
Step 9: generated ' of'
Step 10: generated ' hurdles'
Step 11: generated ' to'
Step 12: generated ' overcome'
Step 13: generated '.'
Step 14: generated ' Here'
Step 15: generated '’s'


In [4]:
import torch.nn.functional as F

for i, (token_id, step_scores) in enumerate(zip(generated_ids, output.scores)):
    probs = F.softmax(step_scores[0], dim=-1)
    chosen_prob = probs[token_id].item()
    token_str = tokenizer.decode(token_id)
    print(f"Step {i+1}: '{token_str}' (confidence: {chosen_prob:.2%})")

Step 1: ' bright' (confidence: 45.79%)
Step 2: ',' (confidence: 52.18%)
Step 3: ' but' (confidence: 91.06%)
Step 4: ' there' (confidence: 11.52%)
Step 5: ' are' (confidence: 85.64%)
Step 6: ' still' (confidence: 27.90%)
Step 7: ' a' (confidence: 19.41%)
Step 8: ' lot' (confidence: 41.17%)
Step 9: ' of' (confidence: 100.00%)
Step 10: ' hurdles' (confidence: 6.90%)
Step 11: ' to' (confidence: 100.00%)
Step 12: ' overcome' (confidence: 100.00%)
Step 13: '.' (confidence: 91.06%)
Step 14: ' Here' (confidence: 11.51%)
Step 15: '’s' (confidence: 58.83%)


In [6]:

from transformers import pipeline
asr = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    device=0
)

result = asr("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac")
print(result["text"])

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

 I have a dream that one day this nation will rise up and live out the true meaning of its creed.


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=quant_config,
    device_map="cuda"
)

transcript = result["text"]

messages = [
    {"role": "system", "content": "You are an assistant that writes concise meeting minutes from transcripts."},
    {"role": "user", "content": f"Summarize this into meeting minutes with key points:\n\n{transcript}"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=200)

summary = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(summary)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

**Meeting Summary:**

- **Date:** [Insert Date]
- **Time:** [Insert Time]
- **Location:** [Insert Location]

**Agenda:**
1. Discussing a vision for societal progress.

**Minutes:**
- Member A expressed a desire for racial equality and justice, stating "I have a dream that one day this nation will rise up and live out the true meaning of its creed."
- Member B acknowledged the significance of such a vision but noted challenges in achieving it.
- Member C proposed creating a task force to explore ways to advance civil rights and social equity within the community.
- Member D suggested holding public forums to gather input on how to best achieve this goal.
- The group agreed to further discuss these ideas during the next meeting to develop a comprehensive plan.


In [8]:
messages = [
    {"role": "system", "content": "You generate realistic synthetic data for testing purposes."},
    {"role": "user", "content": "Generate 5 fictional customer support tickets as a JSON list. Each ticket should have: id, customer_name, issue_summary, priority (low/medium/high)."}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=400)

synthetic_data = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(synthetic_data)

```json
[
    {
        "id": "123456",
        "customer_name": "John Doe",
        "issue_summary": "Product A does not work properly on my device.",
        "priority": "high"
    },
    {
        "id": "789012",
        "customer_name": "Jane Smith",
        "issue_summary": "Service B is slow and unreliable during peak hours.",
        "priority": "medium"
    },
    {
        "id": "567890",
        "customer_name": "Michael Johnson",
        "issue_summary": "Issue C cannot be resolved within the warranty period.",
        "priority": "low"
    },
    {
        "id": "456789",
        "customer_name": "Emily Davis",
        "issue_summary": "Feature D was never implemented in the latest update.",
        "priority": "high"
    },
    {
        "id": "987654",
        "customer_name": "David Brown",
        "issue_summary": "The website E is crashing frequently when I log in.",
        "priority": "medium"
    }
]
```
